# Stage 06: Global Feature Extraction

**Status:** Implemented and unit-tested (`swin_transformer.py`'s `create_dual_scale_swin_model()`
and `GlobalFeatureExtractionStage`). **Not trained, not frozen.** This notebook builds the Stage
06 model, runs it against a small real-data demo batch, and verifies its output shape. It does
**not** start real training -- see Section 7 (`RUN_TRAINING`).

## Objective

Extract a whole-image, long-range, segmentation-independent global representation (Dual-Scale
Swin Transformer -- two parallel Swin branches at patch sizes 4 and 8, fused by channel-wise
concatenation only) for later consumption by Stage 07 (Adaptive Cross-Attention). See
`PROJECT_STRUCTURE.md` Sec 6 and the Stage 06 design-resolution record for the full specification
and its literature/engineering traceability.

## Expected Inputs

- Stage 02 processed RGB only, resized to 256x256 -- **never** Stage 05's `local_features`,
  **never** vessel or lesion probability maps. Stage 06 has no dependency on Stage 03 or Stage 04
  at all.

## Expected Outputs

- `global_features`, shape `(64, 1152)` -- an un-pooled token sequence (8x8 grid flattened),
  never globally pooled, for Stage 07 (Adaptive Cross-Attention).

## Datasets

APTOS 2019 only -- the same corpus and split as Stage 05's own loader
(`global_feature_extraction_dataset.py` reuses `local_feature_extraction_dataset.py`'s
Stage-02-application and split helpers directly, rather than duplicating them). IDRiD and EyeQ
have no role here.

## Dependencies

Stage 02 (Image Preprocessing) only.

## Training Status

Stage 06 has **no standalone training procedure** -- it has no independent global-feature ground
truth of its own. It will eventually be trained jointly with Stage 05 (Local Feature Extraction),
Stage 07 (Adaptive Cross-Attention), RACAF, and Stage 08 (CORN), through the downstream CORN
ordinal loss, once those stages exist. That joint training script is not implemented yet, and is
out of scope for this notebook.

---

This notebook does not modify Stages 01-05 or RACAF, does not implement Stage 07/RACAF/CORN, and
does not start real training.

### Bootstrap

Same minimal clone + `sys.path` setup every stage notebook needs -- see `colab/common/setup.py`'s module docstring for why this is intentionally duplicated.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)

### Imports

The reusable `colab/common/` infrastructure works the same way it does for every other stage
notebook. `swin_transformer.py` (Stage 06's model) and `global_feature_extraction_dataset.py`
(Stage 06's dataset loader) are now implemented -- imported below.

In [ ]:
import setup

setup_info = setup.setup()

import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)


## 3. Configuration

Stage 06's fixed contract (input/output shapes, branch configuration) plus this notebook's own
demo scope. Values come directly from `swin_transformer.py`'s own module-level constants -- not
re-declared or overridden here, so this notebook can never silently drift from the approved
architecture.

In [ ]:
import numpy as np

import config
import global_feature_extraction_dataset as gfed
import swin_transformer as swin

IMAGE_SIZE = swin.DEFAULT_GLOBAL_FEATURE_INPUT_SHAPE[:2]  # (256, 256)
NUM_CHANNELS = gfed.NUM_CHANNELS                            # 3 (RGB only)
OUTPUT_TOKENS = swin.DUAL_SCALE_OUTPUT_TOKENS                # 64
OUTPUT_CHANNELS = swin.DUAL_SCALE_OUTPUT_CHANNELS            # 1152

# This notebook only ever builds a SMALL real-data demo subset -- staging and
# processing the full 3662-image labeled APTOS set is the future joint Stage
# 05-08 training script's job, not this sanity-check notebook's. See Section 5.
DEMO_SAMPLE_COUNT = 6

print(f"Stage 06 input shape:  (B, {IMAGE_SIZE[0]}, {IMAGE_SIZE[1]}, {NUM_CHANNELS})")
print(f"Stage 06 output shape: (B, {OUTPUT_TOKENS}, {OUTPUT_CHANNELS})")
print(f"Branch A: {swin.DUAL_SCALE_BRANCH_A_CONFIG}")
print(f"Branch B: {swin.DUAL_SCALE_BRANCH_B_CONFIG}")
print(f"Demo sample count for this notebook: {DEMO_SAMPLE_COUNT} (of 3662 labeled APTOS images total)")


## 4. APTOS 2019 Demo Subset Staging

Stages the full `train.csv` (small) plus only `DEMO_SAMPLE_COUNT` images (of 3662 total labeled
images) from Drive -- enough for a real, non-synthetic sanity forward pass without this notebook
processing the full dataset. Reuses the exact same staging target
(`local_feature_extraction_dataset.DEFAULT_TRAIN_CSV`/`DEFAULT_TRAIN_IMAGE_DIR`, which
`global_feature_extraction_dataset` re-exports) Stage 05's own notebook already stages to --
running this cell after Stage 05's notebook (or after this cell in a prior session) reuses the
same local files rather than re-copying them.

In [ ]:
import csv
import posixpath
import shutil

APTOS_TRAIN_CSV_DRIVE = posixpath.join(colab_config.APTOS2019_DATASET_DIR, "raw", "train.csv")
APTOS_TRAIN_IMAGES_DRIVE = posixpath.join(colab_config.APTOS2019_DATASET_DIR, "raw", "train_images")

os.makedirs(os.path.dirname(gfed.DEFAULT_TRAIN_CSV), exist_ok=True)
os.makedirs(gfed.DEFAULT_TRAIN_IMAGE_DIR, exist_ok=True)

if not os.path.isfile(gfed.DEFAULT_TRAIN_CSV):
    if not os.path.isfile(APTOS_TRAIN_CSV_DRIVE):
        raise RuntimeError(f"APTOS2019 train.csv not found on Drive: {APTOS_TRAIN_CSV_DRIVE}.")
    shutil.copy2(APTOS_TRAIN_CSV_DRIVE, gfed.DEFAULT_TRAIN_CSV)

with open(gfed.DEFAULT_TRAIN_CSV, newline="", encoding="utf-8") as f:
    demo_rows = list(csv.DictReader(f))[:DEMO_SAMPLE_COUNT]

for row in demo_rows:
    filename = f"{row['id_code']}.png"
    dst = os.path.join(gfed.DEFAULT_TRAIN_IMAGE_DIR, filename)
    if os.path.isfile(dst):
        continue  # already staged (e.g. by Stage 05's notebook)
    src = posixpath.join(APTOS_TRAIN_IMAGES_DRIVE, filename)
    if not os.path.isfile(src):
        raise RuntimeError(f"APTOS2019 image not found on Drive: {src}.")
    shutil.copy2(src, dst)

print(f"train.csv + {len(demo_rows)} demo images available under {gfed.DEFAULT_TRAIN_IMAGE_DIR}")
print(f"Demo id_codes / diagnoses: {[(r['id_code'], r['diagnosis']) for r in demo_rows]}")


## 5. Stage 06 Input Construction

Builds one real `(256, 256, 3)` Stage 06 input tensor per demo image, via
`global_feature_extraction_dataset._build_sample` -- Stage 02 preprocessing applied live (reusing
`local_feature_extraction_dataset`'s existing helpers, unmodified). No vessel/lesion channel, no
ground-truth mask of any kind is read for these images.

In [ ]:
demo_inputs = []
demo_labels = []
for row in demo_rows:
    id_code = row["id_code"]
    diagnosis = int(row["diagnosis"])
    x, y = gfed._build_sample(id_code, diagnosis, gfed.DEFAULT_TRAIN_IMAGE_DIR, image_size=IMAGE_SIZE)
    demo_inputs.append(x)
    demo_labels.append(y)

demo_batch = np.stack(demo_inputs, axis=0)
print(f"Demo batch shape: {demo_batch.shape}  (dtype={demo_batch.dtype})")
print(f"Value range: [{demo_batch.min():.3f}, {demo_batch.max():.3f}]")
print(f"Demo labels (APTOS DR grade 0-4): {demo_labels}")


## 6. Stage 06 Model Construction + Sanity Forward Pass

Builds the Dual-Scale Swin Transformer via `create_dual_scale_swin_model()` -- random
initialization, no ImageNet weights, **uncompiled** (Stage 06 has no standalone loss). Not
trained. Runs the real demo batch from Section 5 through the freshly built, untrained model --
this verifies the `(256, 256, 3) -> (64, 1152)` contract end-to-end on real (not synthetic) data;
the resulting features are meaningless (random weights), which is expected and fine for a
plumbing check.

In [ ]:
global_feature_model = swin.create_dual_scale_swin_model()
global_feature_model.summary()

total_params = global_feature_model.count_params()
trainable_params = int(sum(np.prod(v.shape) for v in global_feature_model.trainable_variables))
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Compiled with a loss: {global_feature_model.loss is not None} (must be False)")

demo_output = global_feature_model.predict(demo_batch, verbose=0)
print(f"\nInput shape:  {demo_batch.shape}")
print(f"Output shape: {demo_output.shape}")
print(f"Output is finite: {np.isfinite(demo_output).all()}")

expected_output_shape = (demo_batch.shape[0], OUTPUT_TOKENS, OUTPUT_CHANNELS)
assert demo_output.shape == expected_output_shape, (
    f"Expected {expected_output_shape}, got {demo_output.shape}"
)
print(f"\nMatches the approved output contract: (B, {OUTPUT_TOKENS}, {OUTPUT_CHANNELS})")


### Checkpoint save/load sanity check

Confirms the model round-trips through weights-only save/load with identical predictions -- a
plumbing check only. Stage 06's custom Swin layer classes (PatchEmbed, WindowAttention,
SwinTransformerBlock, PatchMerging, BasicLayer -- all pre-existing, unmodified) have no
`get_config()` implementation, so `GlobalFeatureExtractionStage` persists weights only
(`.weights.h5`), not a full-architecture `.keras` file: rebuild via `create_dual_scale_swin_model()`,
then `load_weights()` -- see `GlobalFeatureExtractionStage.save()`'s docstring. This is not a
deviation from project convention -- `training/callbacks.py`/`training/trainer.py`'s shared
framework defaults to `save_weights_only=True` (`.weights.h5`); Stage 04 is the one that opts into
full `.keras` checkpoints. No real checkpoint is produced by this notebook (RUN_TRAINING stays
False, Section 7) -- this saves to a temporary local path and does not touch Drive or any
exported-model location.


In [ ]:
with tempfile.TemporaryDirectory() as tmp_dir:
    checkpoint_path = os.path.join(tmp_dir, "sanity_check.weights.h5")
    global_feature_model.save_weights(checkpoint_path)

    reloaded_model = swin.create_dual_scale_swin_model()
    reloaded_model.load_weights(checkpoint_path)
    reloaded_output = reloaded_model.predict(demo_batch, verbose=0)

    np.testing.assert_allclose(demo_output, reloaded_output, atol=1e-5)
    print("Save/load sanity check passed: reloaded model produces identical predictions.")


## 7. Training Status

**`RUN_TRAINING` stays `False` in this notebook, following the same project-standard
"prepared, not auto-started" convention every other stage notebook uses.** Unlike Stage 04's
notebook, there is nothing this flag could start even if set to `True` -- Stage 06 has no
standalone training procedure (`GlobalFeatureExtractionStage.train()` raises
`NotImplementedError` by design). Real training will happen in a future, not-yet-implemented
joint Stage 05-08 + RACAF training script, once Stage 07 exists.

In [ ]:
RUN_TRAINING = False  # Stage 06 has no standalone training procedure -- see this section's markdown.

if not RUN_TRAINING:
    print("RUN_TRAINING is False -- no training was started in this notebook.")
    print("Stage 06 is IMPLEMENTED and TESTED, but NOT TRAINED and NOT FROZEN.")
else:
    raise RuntimeError(
        "Stage 06 has no standalone training procedure to run -- see "
        "GlobalFeatureExtractionStage.train()'s NotImplementedError. Real training happens only "
        "as part of the future joint Stage 05-08 + RACAF training script."
    )


## 8. Summary

In [ ]:
print("=" * 72)
print("Stage 06: Global Feature Extraction -- Summary")
print("=" * 72)
print("Architecture: Dual-Scale Swin Transformer -- two parallel branches")
print(f"  Branch A (fine):   patch_size={swin.DUAL_SCALE_BRANCH_A_CONFIG['patch_size']}, "
      f"depths={swin.DUAL_SCALE_BRANCH_A_CONFIG['depths']}, "
      f"heads={swin.DUAL_SCALE_BRANCH_A_CONFIG['num_heads']}")
print(f"  Branch B (coarse): patch_size={swin.DUAL_SCALE_BRANCH_B_CONFIG['patch_size']}, "
      f"depths={swin.DUAL_SCALE_BRANCH_B_CONFIG['depths']}, "
      f"heads={swin.DUAL_SCALE_BRANCH_B_CONFIG['num_heads']}")
print("  Fusion: channel-wise concatenation only -- no projection, no cross-attention")
print(f"Input:  (B, {IMAGE_SIZE[0]}, {IMAGE_SIZE[1]}, {NUM_CHANNELS}) -- Stage 02 RGB only")
print(f"Output: (B, {OUTPUT_TOKENS}, {OUTPUT_CHANNELS}) -- un-pooled token sequence")
print(f"Total parameters: {total_params:,} (trainable: {trainable_params:,})")
print(f"Demo forward pass: input {demo_batch.shape} -> output {demo_output.shape}")
print(f"\nFrozen dependencies used (unmodified): Stage 02 preprocessing only -- "
      "Stage 06 has no dependency on Stage 03 or Stage 04.")
print(f"\nSTATUS: IMPLEMENTED, TESTED, NOT TRAINED, NOT FROZEN.")
print("Training will be a separate step, after Stage 07 exists and this implementation is reviewed.")
print("\nNovelty status: not a research contribution -- Swin's core mechanism, the dual-patch-")
print("size branch concept, and the (4,8) pairing are established/cited (see the Stage 06 design-")
print("resolution record). RACAF remains this project's sole research innovation.")
